In [ ]:
import sys
sys.path.append('..')

from src.flow_matching.distributions import Gaussian, CheckerboardSampleable
from src.flow_matching.probability_path import LinearConditionalProbabilityPath
from src.flow_matching.training import ConditionalFlowMatchingTrainer
from src.flow_matching.integration import EulerODESolver, ODEBackwardSolver
from src.flow_matching.models import MLPVectorField
from src.helpers import record_every
from src.flow_matching.plotting import hist2d_samples
from src.flow_matching.helpers import choose_device

from matplotlib import pyplot as plt
import torch

device = choose_device()

# Example from MIT Lab two

Modified example from the MIT course where isotropic Gaussian is transformed into an arbitrary distribution via a linear conditional probability path, comparing the ground-truth path and the path resulting from a learned vector field. 

In [ ]:
# Construct conditional probability path
path = LinearConditionalProbabilityPath(
    p_data = CheckerboardSampleable(device, grid_size=4),
    p_simple = Gaussian.isotropic(dim=2, std=1, device=device)
).to(device)

# Construct learnable vector field
vector_field = MLPVectorField(dim=2, hiddens=[64,64,64,64])

# Construct trainer
trainer = ConditionalFlowMatchingTrainer(path, vector_field)
losses = trainer.train(num_epochs=5000, device=device, lr=1e-3, batch_size=2000, save_checkpoint=False, fixed_N=True)

In [ ]:
vector_field.eval()

In [ ]:
num_marginals = 5    # number of snapshots to plot
num_samples = 50000  # number of samples per snapshot 

plt.figure(figsize=(20,8))

# -------------------------------------------- #
#        Plotting Ground-Truth Marginals       #
# -------------------------------------------- #

ts = torch.linspace(0.0, 1.0, num_marginals).to(device)

for idx, t in enumerate(ts):

    tt = t.view(1,1).expand(num_samples, 1)
    xts = path.sample_marginal_path(tt)

    plt.subplot(2, num_marginals, idx + 1)
    hist2d_samples(xts.cpu(), range=[[-5, 5], [-5, 5]])

    plt.xticks([])
    plt.yticks([])
    plt.title(f'$t={t.item():.2f}$', fontsize=15)

    plt.ylabel("Ground Truth", fontsize=20) if idx == 0 else None

# --------------------------------------------  #
#  Plotting Marginals of Learned Vector Field   #
# -------------------------------------------   #

# initialize ODE solver
simulator = EulerODESolver(vector_field)
ts = torch.linspace(0,1,100).to(device)

# solve / simulate ODE from x0
x0 = path.p_simple.sample(num_samples)
xts = simulator.solve_with_trajectory(x0, ts.view(1,-1,1).expand(num_samples,-1,1))

# save only num_marginal snapshots of the trajectory
record_every_idxs = record_every(len(ts), len(ts) // (num_marginals - 1))
xts = xts[:,record_every_idxs,:]

plot_idx = idx + 1

# plot the snapshots
for idx in range(xts.shape[1]):
    xx = xts[:,idx,:]

    plt.subplot(2, num_marginals, plot_idx + idx + 1)
    hist2d_samples(xx.cpu(), range=[[-5, 5], [-5, 5]])

    tt = ts[record_every_idxs[idx]]

    plt.xticks([])
    plt.yticks([])
    plt.title(f'$t={tt.item():.2f}$', fontsize=15)
    plt.ylabel("Learned", fontsize=20) if idx == 0 else None

plt.show()


In [ ]:
plt.hist2d(xx.cpu()[:,0],xx.cpu()[:, 1], bins=100, cmin=2)
plt.axis('equal')

# Testing calculating log prob of final distribution

By using the formula for the posterior probability in CNF methods, we can evaluate the probability values  $p_1(\theta_1)$ of samples $\theta_1$. We do this by integrating two ODEs numerically, backwards.

$$
x_{1-t} = x_t - u_t(x_t)\Delta t \\
f(1-t) = f(t) + \nabla u_{t-1}(x_{t-1})
$$

where we start with initial conditions $x_1\sim p(\theta)$ is given/generated (point that we want to know the probability value of) and $f(1) = 0$. Since $f(0) = \log p_0(\theta_0) - \log p_1(\theta_1)$, we can evaluate $\log p_1$ with the result;
$$
\log p_1 = \log p_0 - f(0)
$$

In [ ]:
simulator = ODEBackwardSolver(vector_field)

# integrate backwards
ts = torch.linspace(1,0,100, device=device) 
num_samples = 1000
indices = torch.randint(low=0, high=xts.shape[0], size=(num_samples,))

# TODO: check if first of xts, and x_init identical, can maybe re-use saved trajectory instead of integratin backwards
# meh, probably too memory intensive later on, better to just do it iteratively.
x_final = xts[indices, -1, :].clone().detach().requires_grad_(True)
x_init, f_init = simulator.solve(x_final, ts.view(1,-1,1).expand(num_samples,-1,1))

In [ ]:
# plot to check initial distribution matches expectation
hist2d_samples(x_init.detach().cpu(), range=[[-5, 5], [-5, 5]])

plt.xticks([])
plt.yticks([])
plt.title(f'init distr found through backwards Euler', fontsize=15)
plt.show()

In [ ]:
# find probability values of generated samples
log_prob = simulator.f_to_posterior_log_prob(f_init, x_init)

In [ ]:
max_idx = torch.argmax(log_prob)
max_prob = log_prob[max_idx]
max_point = x_final[max_idx]

print(
    "The MPE (maximum probability estimate) is: ", max_point.cpu().detach().numpy(), 
    f"\n \n with a probability of: {torch.exp(max_prob).item():.5f}" 
    )
torch.sum(torch.exp(log_prob))

top_probs, top_idxs = torch.topk(log_prob, 800)
torch.sum(torch.exp(top_probs) / 18)

In [ ]:
plt.figure(figsize=(10,4))

# plot histogram of samples
plt.subplot(121)
plt.title('Histogram of generated FM samples')
plt.hist2d(xx.cpu()[:,0],xx.cpu()[:, 1], bins=100, cmin=2)
plt.axis('equal')


# plot samples colored by probability value
plt.subplot(122)
plt.title("Log of generated samples indicated by color")
plt.scatter(
    x_final[:,0].detach().cpu(), 
    x_final[:,1].detach().cpu(), 
    c=log_prob.detach().cpu()
    )
plt.colorbar()
plt.show()